# Box Office Forecasting: the decision log

Every modelling choice in this project, in the order it was made, with the
evidence that settled it. Each section states a decision, shows the check that
informed it, and records what was rejected and why.

The thing being predicted is what a film earns **before it opens**, from what is
knowable on the day the marketing campaign starts.

Run top to bottom. Every number below is recomputed, not transcribed.


In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'boxoffice' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))

import numpy as np, pandas as pd
pd.set_option('display.width', 110)
print('project root:', ROOT)

project root: /Users/thomaslappas/Desktop/MovieGame


---
## Decision 1: what counts as leakage

The obvious rule is *do not use the revenue column*. That is not the rule that
matters. The Clean Sweep seed this project sits beside carries eight columns
that all look like ordinary film metadata and all accumulate **after** a film
has been in front of an audience.

Using any of them produces an excellent score and a worthless model, so they
are named and blocked rather than avoided by discipline.


In [2]:
from boxoffice.model.leakage import BANNED, assert_clean, LeakageError

for col, why in list(BANNED.items())[:9]:
    print(f'{col:22s} {why}')

imdb_rating            accumulates from votes cast after release
imdb_votes             accumulates from votes cast after release
rt_critic              aggregate settles after the review embargo lifts
rt_audience            audience score requires an audience
metascore              aggregate settles after the review embargo lifts
nominations            awarded months to years after release
wins                   awarded months to years after release
award_points           derived from nominations and wins
award_standing         derived from nominations and wins


In [3]:
# The guard fires on a matrix that contains one.
bad = pd.DataFrame({'log_budget': [1.0], 'imdb_rating': [7.2]})
try:
    assert_clean(bad)
except LeakageError as exc:
    print('blocked ->', exc)

blocked -> post-release columns in the feature matrix -> imdb_rating: accumulates from votes cast after release


---
## Decision 2: the leak that survives code review

A director's prior average gross is a legitimate feature. The same average
computed over their **whole career**, including films that had not opened yet,
is not, and the column name is identical either way. A plain `groupby` gives
you the second one.

`AsOf` sorts by release date and lets each row see only strictly earlier rows.
The case worth testing is two films opening the same day: they must both see
the history as of that morning, and neither may see the other.


In [4]:
from boxoffice.model.leakage import AsOf

demo = pd.DataFrame({
    'title':        ['A', 'B', 'C', 'D'],
    'director_id':  ['d1'] * 4,
    'release_date': pd.to_datetime(['2010-01-01','2012-01-01','2014-06-01','2014-06-01']),
    'log_ww':       [10.0, 20.0, 99.0, 99.0],
})
demo['prior_median'] = AsOf('director_id', 'log_ww').transform(demo, 'median')
demo

,title,director_id,release_date,log_ww,prior_median
0,A,d1,2010-01-01,10.0,NaN
1,B,d1,2012-01-01,20.0,10.0
2,C,d1,2014-06-01,99.0,15.0
3,D,d1,2014-06-01,99.0,15.0


In [5]:
prior = demo['prior_median']
checks = {
    'first film has no history':      pd.isna(prior.iloc[0]),
    'B sees only A':                  prior.iloc[1] == 10.0,
    'C sees A and B':                 prior.iloc[2] == 15.0,
    'D, same day as C, sees the same': prior.iloc[3] == 15.0,
    'neither sees the other s 99':    99.0 not in set(prior.dropna()),
}
for name, ok in checks.items():
    print(f'{"PASS" if ok else "FAIL"}  {name}')

PASS  first film has no history
PASS  B sees only A
PASS  C sees A and B
PASS  D, same day as C, sees the same
PASS  neither sees the other s 99


---
## Decision 3: the sampling frame

The population is films that were financed and given a theatrical release, so
the frame has to be defined by pre-release facts only.

TMDB tags almost everything theatrical and returns **37,486 titles for 2019
alone**, nearly all festival and direct-to-video tail with no budget and no
gross. Every convenient way to cut that down selects on the outcome:

| Filter | Verdict |
|---|---|
| Sort by revenue | rejected: selects on the target |
| Filter by vote count | rejected: selects on who eventually saw it |
| Filter by popularity | rejected: the same, measured today |
| **Filter by distributor** | **accepted: known months before release** |

A distributor is attached long before opening weekend, so anchoring on one
keeps the sample definition itself free of the answer.


In [6]:
from boxoffice.pipeline.fetch import DISTRIBUTORS, BUDGET_FLOOR

feat = pd.read_parquet(ROOT / 'backend/boxoffice/data/features.parquet')
print(f'{len(DISTRIBUTORS)} distributors in the frame, budget floor ${BUDGET_FLOOR:,}')
print(f'{len(feat):,} films, {feat.release_date.dt.year.min()}-{feat.release_date.dt.year.max()}')
feat.release_date.dt.year.value_counts().sort_index().tail(8).to_frame('films')

43 distributors in the frame, budget floor $1,000,000
2,457 films, 2000-2025


,films
release_date,
2018,114
2019,63
2020,36
2021,64
2022,70
2023,76
2024,64
2025,75


---
## Decision 4: no imputation

Half the sample has a director with no earlier film in it. That is not missing
data to be filled; it is a fact about the film. Imputing the median would tell
the model every debut is average.

`HistGradientBoostingRegressor` splits on missingness directly, so *unknown*
stays a category rather than becoming a guess.


In [7]:
cols = [c for c in feat.columns if not c.startswith('y_')
        and c not in ('film_key','title','imdb_id','release_date')]
cov = feat[cols].notna().mean().sort_values()
cov.head(8).to_frame('coverage').style.format('{:.0%}') if hasattr(cov, 'style') else cov.head(8)

director_prior_max_log              0.497761
director_prior_median_log           0.497761
cinematographer_prior_median_log    0.674807
cinematographer_prior_max_log       0.674807
composer_prior_max_log              0.737078
composer_prior_median_log           0.737078
composer_prior_count                0.944648
cast_prior_max_log                  0.953195
dtype: float64

---
## Decision 5: the bar to clear

Not R-squared, and not the median film. Box office spans five orders of
magnitude, so squared error in dollars is decided by half a dozen titles, and
the median is a bar nobody would accept.

**Budget alone is the real baseline**: what a studio spends is most of what a
studio makes. The headline metric is the share of forecasts landing within a
factor of two, because that is how a forecast gets used.

Folds are strictly temporal. Random K-fold would let the model learn 2019 from
2021, which is the second most common way these models get flattered.


In [8]:
from boxoffice.model.train import run, summarise, feature_columns

folds = run(feat, feature_columns(feat))
print(f'{len(folds)} rolling-origin folds, {folds[0].year}-{folds[-1].year}')
summarise(folds)

16 rolling-origin folds, 2010-2025


,estimator,mae_log,within_2x
0,median,1.477540,0.310021
1,budget_only,1.052722,0.473414
2,model,0.947880,0.557341


In [9]:
beat = sum(f.within_2x['model'] > f.within_2x['budget_only'] for f in folds)
print(f'model beats budget-only in {beat} of {len(folds)} folds')
pd.DataFrame([{'year': f.year, 'n': f.n_test,
               'model': f.within_2x['model'],
               'budget_only': f.within_2x['budget_only']} for f in folds])

model beats budget-only in 15 of 16 folds


,year,n,model,budget_only
0,2010,104,0.634615,0.605769
1,2011,105,0.580952,0.590476
2,2012,90,0.588889,0.466667
3,2013,96,0.687500,0.593750
4,2014,99,0.595960,0.525253
5,2015,106,0.594340,0.452830
6,2016,116,0.646552,0.525862
7,2017,100,0.580000,0.510000
8,2018,114,0.561404,0.482456
9,2019,63,0.412698,0.238095


---
## Decision 6: which ablation answers the question

The first design added each feature group to a budget-only model. Almost every
group came back *harmful* -- yet all 35 features together gained eleven points.

Two things cause that and only one is interesting. The real one is that these
features only mean something in combination: a star's prior gross says nothing
until you know the budget tier and genre attached to it. The other is a
confound I introduced by holding the boosting configuration fixed across very
different feature counts, so five features and four hundred iterations overfit
in a way thirty-five do not.

Adding a group to a one-feature baseline tests whether it can carry a model
alone. Nobody asked that. **Removing it from the working model** tests what it
contributes alongside the others, which is how it will be used.


In [10]:
abl = pd.read_csv(ROOT / 'backend/boxoffice/data/ablation.csv')
add = abl[abl.design == 'add one to budget'][['group','d_mae','d_2x']]
loo = abl[abl.design == 'remove one from full'][['group','d_mae','d_2x']]
print('ADD ONE TO BUDGET (negative d_mae = helps)'); display(add)
print('REMOVE ONE FROM FULL (positive d_mae = it was carrying weight)'); display(loo)

ADD ONE TO BUDGET (negative d_mae = helps)


,group,d_mae,d_2x
0,budget only (baseline),0.000000,0.000000
1,+ form,-0.054452,0.061872
2,+ genre,0.018042,-0.005157
3,+ calendar,0.046049,-0.004407
4,+ studio,0.000980,0.017321
5,+ director,0.011330,0.011937
6,+ cast,0.075605,-0.010399
7,+ cinematographer,0.032307,-0.004044
8,+ composer,0.036760,0.003575
9,all groups,-0.121696,0.110010


REMOVE ONE FROM FULL (positive d_mae = it was carrying weight)


,group,d_mae,d_2x
10,- form,0.088087,-0.067017
11,- genre,0.013035,-0.007256
12,- calendar,0.007841,-0.015117
13,- studio,0.040276,-0.030974
14,- director,-0.006834,0.004231
15,- cast,-0.000884,-0.009140
16,- cinematographer,0.000047,0.003327
17,- composer,0.005581,-0.017347


### What the second design says

Ranked by what the full model loses when the group is removed:

* **Form** (sequel, franchise position, runtime, language) is the largest single
  contributor, at roughly twice the studio.
* **Studio** is second. The distributor carries more than any of the people.
* **Cinematographer lands at exactly zero.** That was the prediction before the
  run, and it is now the data's answer rather than an assertion.
* **Cast contributes nothing measurable**, and removing the **director** very
  slightly improves the model.

What survives is what the film *is*, not who made it.


---
## Decision 7: a claim that did not survive contact

The original design argued that worldwide should be the **sum** of a domestic
fit and an international fit, because the two halves have different drivers.
That is a reasonable-sounding assertion, so it was tested rather than left
standing.

It loses. Each half is fit in log space and exponentiated before summing, so
two independent errors compound instead of cancelling, while a direct fit
optimises the quantity actually wanted. The README was rewritten to say so.

What survived is the other half of the argument: international is the harder
side by roughly ten points.


In [11]:
from boxoffice.model.splits import run as run_split

split = run_split()
print(f"{split.attrs['counts']['usable']} films with both figures")
display(split)
split[['domestic','international','ww_from_split','ww_direct']].mean().to_frame('mean')

686 films with both figures


,year,n,domestic,international,ww_from_split,ww_direct
0,2012,30,0.833333,0.700000,0.800000,0.833333
1,2013,33,0.727273,0.636364,0.727273,0.696970
2,2014,35,0.742857,0.628571,0.742857,0.742857
3,2015,36,0.666667,0.583333,0.694444,0.777778
4,2016,28,0.642857,0.607143,0.678571,0.678571
5,2017,40,0.675000,0.650000,0.700000,0.725000
6,2018,35,0.742857,0.628571,0.714286,0.742857
7,2021,20,0.450000,0.350000,0.450000,0.550000
8,2022,34,0.764706,0.588235,0.735294,0.705882


,mean
domestic,0.693950
international,0.596913
ww_from_split,0.693636
ww_direct,0.717028


---
## Decision 8: what a projection may be shown next to

Fitting one model on everything and printing its prediction beside the real
gross produces a beautiful scatter and grades the model's own memory.

Every released film is instead scored by the model from the fold where its own
year was the test year. The number beside a 2019 film is what a forecaster
standing in December 2018 would have said. Films released before the first
validation fold get **no** projection rather than an in-sample one.


In [12]:
proj = pd.read_parquet(ROOT / 'backend/boxoffice/data/projections.parquet')
scored = proj.dropna(subset=['projected_worldwide'])
print(f'{len(proj):,} films, {len(scored):,} with an out-of-sample projection')
print(f"within a factor of two: {scored.within_2x.mean():.1%}")
print(f"no projection offered:  {proj.projected_worldwide.isna().sum():,}")

show = scored.assign(
    actual=lambda d: (d.actual_worldwide/1e6).round(0),
    projected=lambda d: (d.projected_worldwide/1e6).round(0))
show.nlargest(8, 'actual_worldwide')[['title','actual','projected','ratio']]

2,457 films, 1,378 with an out-of-sample projection
within a factor of two: 57.3%
no projection offered:  1,079


,title,actual,projected,ratio
1676,Star Wars: The Force Awakens,2068.0,741.0,0.358426
1933,Avengers: Infinity War,2052.0,1299.0,0.633046
2166,Spider-Man: No Way Home,1921.0,707.0,0.368032
2346,Inside Out 2,1699.0,661.0,0.389283
1615,Jurassic World,1672.0,513.0,0.307035
1317,The Avengers,1519.0,685.0,0.451036
1597,Furious 7,1515.0,761.0,0.502104
2197,Top Gun: Maverick,1489.0,564.0,0.379074


---
## What is not established

* **2020 breaks.** Both model and baseline collapse. The pandemic severed the
  budget-to-gross relationship and a model trained through 2019 had no way to
  know. That is temporal validation being honest, not a bug.
* **Domestic coverage is 28%** and thinnest in the recent years that matter
  most, so the three-way split above is directional. A daily job is filling it.
* **The sample is studio films.** It will not price a microbudget breakout,
  because the frame excluded films no distributor picked up.
* **Cast contributing nothing may be a sample-size result** rather than a truth
  about stardom. 2,457 films is not many, and prior gross is a crude proxy for
  drawing power.
